### Imports

In [34]:
import pandas as pd
import numpy as np
import os
import albumentations as A
import torch
from torch.utils.data import Dataset, DataLoader, SequentialSampler, RandomSampler
import cv2
import timm
from tqdm.notebook import tqdm
import matplotlib.pyplot
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, SGD, AdamW
from torchvision.ops import sigmoid_focal_loss

### Data

In [10]:
train = pd.read_csv('./neoai-2025-underfitting-cv/train.csv')
train['path'] = [f'./neoai-2025-underfitting-cv/train_images/{x}' for x in train['path']]

paths_list = []
main_path = './neoai-2025-underfitting-cv/test_images'
for path in sorted(os.listdir(main_path)):
    paths_list += [f'{main_path}/{path}']

test = pd.DataFrame()
test['path'] = paths_list
test['class'] = 0

train['class'].value_counts()

class
4     10
53    10
75    10
43    10
54    10
92    10
56    10
68    10
28    10
87    10
85    10
19    10
8     10
38    10
12    10
32    10
14    10
78    10
24    10
13    10
36    10
44    10
5     10
Name: count, dtype: int64

In [80]:
class TrainDataset(Dataset):
    def __init__(self, path, target, transform):
        self.path = path
        self.target = target
        self.transform = transform
    def __len__(self):
        return len(self.target)

    def __getitem__(self, item):

        image = cv2.imread(self.path[item])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
        target = self.target[item]
        image = self.transform(image=image)['image']
        image = image.astype(np.float32) / 255.0
        image = image - 0.5
        image = torch.from_numpy(image).permute(2, 0, 1)

        return image, target


def get_train_transforms(dim = 224):
    return A.Compose(
        [
            A.LongestMaxSize(max_size=dim, p=1.0),
            A.PadIfNeeded(dim, dim, p = 1.0),
            A.HorizontalFlip(p = 0.5),
            A.ShiftScaleRotate(
                shift_limit=0.1,
                scale_limit=0.2,
                rotate_limit=30,
                p=0.7
            )
        ]
  )

def get_valid_transforms(dim = 224):
    return A.Compose(
        [
            A.LongestMaxSize (max_size=dim, p=1.0),
            A.PadIfNeeded(dim, dim, p = 1.0),
        ]
  )

### Model

In [66]:
class PetNet(nn.Module):
    def __init__( self, model_name, num_classes):
        super().__init__()
        self.model = timm.create_model(model_name, num_classes=num_classes)

    def forward(self, image):
        x = self.model(image)
        return x

In [81]:
batch_size = 32
params_train = {'batch_size': batch_size, 'shuffle': True, 'drop_last': False, 'num_workers': 0}
params_valid = {'batch_size': batch_size, 'shuffle': False, 'drop_last': False, 'num_workers': 0}
device = 'cuda'
dim = 224

train_loader = DataLoader( TrainDataset( train['path'].tolist(), train['class'].tolist(), get_train_transforms(dim) ), **params_train)
valid_loader = DataLoader( TrainDataset( test['path'].tolist(), test['class'].tolist(), get_valid_transforms(dim) ), **params_valid)

model = PetNet("tiny_vit_5m_224.dist_in22k_ft_in1k", num_classes = 102)

model_dict = torch.load("./neoai-2025-underfitting-cv/model.pt", map_location='cuda', weights_only = False)

In [46]:
bad_classes = [0.5 if i in train['class'].to_numpy() else 0.1 for i in range(102)]
weights = torch.tensor(bad_classes, dtype=torch.float, device=device)

In [77]:
class FocalLoss(nn.Module):
    def __init__(self, weights, gamma):
        super().__init__()
        self.weights = weights
        self.gamma = gamma
    
    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, self.weights, reduction='none')
        p_softmax = torch.exp(-ce_loss)
        focal_loss = ((1-p_softmax)**self.gamma) * ce_loss

        return focal_loss.mean()

In [82]:
epochs = 110
lr = 2e-5

model.load_state_dict(model_dict, strict=False)
model = model.to(device)
model.train()

optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
criterion = FocalLoss(weights, gamma=0.5)
scaler = torch.amp.GradScaler('cuda')
clip_grad_norm = 0.9

for epoch in range(epochs):
    len_dataloader = len(train_loader)
    average_loss = 0
    tk0 = tqdm(enumerate(train_loader), total = len_dataloader)
    for batch_number,  (inputs, labels)  in tk0:

        optimizer.zero_grad()
        inputs = inputs.cuda()
        labels = labels.cuda().long()

        with torch.amp.autocast('cuda'):
            y_preds  = model(inputs)
            loss = criterion(y_preds, labels)

        scaler.scale(loss).backward()

        if clip_grad_norm > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        average_loss += loss.cpu().detach().numpy()
        tk0.set_postfix(loss=average_loss / (batch_number + 1), stage="train", epoch = epoch)

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

...

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

### Test

In [83]:
def make_predict(state_dict, valid_loader, name_csv = 'submission.csv', test_ids = [x.split('/')[-1] for x in test['path']]):
    preds = []
    len_loader = len(valid_loader)
    tk0 = tqdm(enumerate(valid_loader), total = len_loader)
    average_loss = 0
    model = timm.create_model( "tiny_vit_5m_224.dist_in22k_ft_in1k", num_classes=102)
    model.cuda().eval()
    model.load_state_dict(state_dict)
    
    with torch.no_grad():
        for batch_number,  (inputs, labels)  in tk0:
            inputs = inputs.cuda()
            labels = labels.cuda().long()
    
            with torch.amp.autocast('cuda'):
                y_preds  = model(inputs)
    
            preds += [y_preds.to('cpu').numpy()]
    
    preds = np.concatenate(preds)    

    model.train()

    submission = pd.DataFrame()
    submission['id'] = test_ids
    submission['class'] = np.argmax(preds, 1)
    submission.to_csv(name_csv, index = None)

In [84]:
make_predict(model.model.state_dict(), valid_loader, name_csv='sumbission_20_no_aug.csv')

  0%|          | 0/160 [00:00<?, ?it/s]